# quadruped_rl_sim -- entrenamiento en la nube (Colab)

Corre el mismo `train.py` del repo, pero en los servidores de Colab en vez de tu laptop.

**Importante sobre velocidad**: este entrenamiento es CPU-bound (el cuello de botella es la
simulacion de fisica de PyBullet, no la red neuronal), asi que la GPU gratis de Colab **no**
acelera esto. Anda a `Entorno de ejecucion > Cambiar tipo de entorno` y dejalo en **CPU** (sin
acelerador) -- consume menos cuota y da igual. Lo que si importa es la cantidad de vCPUs
(el tier gratis suele dar 2).

**Persistencia**: todo corre clonado DENTRO de Google Drive, asi los checkpoints sobreviven
aunque Colab te desconecte (pasa solo, cada ~12h o por inactividad). Cada vez que vuelvas,
corre las celdas de arriba a abajo de nuevo -- la celda de entrenamiento detecta el ultimo
checkpoint guardado y sigue desde ahi (usa el `--resume` de `train.py`), no arranca de cero.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/abdair-coca/BipedalRobot.git"
PROJECT_DIR = "/content/drive/MyDrive/BipedalRobot"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}/quadruped_rl_sim

Instala las dependencias pineadas del proyecto. En Linux `pybullet` SI tiene wheel
precompilado (a diferencia de Windows), asi que esto deberia ser rapido, sin compilar nada.

Si Colab te pide reiniciar el entorno de ejecucion despues de este paso (por el cambio de
version de numpy), hacelo y volve a correr desde la celda de Drive.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python generate_urdf.py

Tensorboard inline (se actualiza solo mientras la celda de entrenamiento de abajo corre en
otra celda -- Colab permite correr celdas en paralelo).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## Entrenar (con auto-resume)

Corre en tandas de `TIMESTEPS_POR_TANDA` steps. Si ya hay un checkpoint guardado en Drive
(de una corrida anterior, tuya o de una sesion cortada), sigue desde ahi -- no pisa nada,
no arranca de cero. Volve a correr esta celda las veces que quieras, incluso en dias
distintos: cada vez continua el entrenamiento total.

In [ ]:
import glob
import os
import re

TIMESTEPS_POR_TANDA = 500_000
RUN_NAME = "ppo_quadruped"

n_envs = max(1, os.cpu_count() or 2)
print(f"Usando {n_envs} entornos en paralelo (segun vCPUs disponibles)")


def latest_checkpoint(ckpt_dir="checkpoints", prefix=RUN_NAME):
    files = glob.glob(os.path.join(ckpt_dir, f"{prefix}_*_steps.zip"))
    if not files:
        return None
    files.sort(key=lambda f: int(re.search(r"_(\d+)_steps\.zip$", f).group(1)))
    return files[-1]


ckpt = latest_checkpoint()
if ckpt:
    print("Resumiendo desde:", ckpt)
    !python train.py --resume "{ckpt}" --timesteps {TIMESTEPS_POR_TANDA} --n-envs {n_envs} --run-name {RUN_NAME}
else:
    print("No hay checkpoint previo, arrancando entrenamiento nuevo")
    !python train.py --timesteps {TIMESTEPS_POR_TANDA} --n-envs {n_envs} --run-name {RUN_NAME}

## Ver el gait sin GUI (Colab no tiene pantalla)

`enjoy.py` necesita una ventana (modo `p.GUI`), que no existe en Colab. Esta celda en cambio
corre la politica en modo headless (`p.DIRECT`) y graba los frames a un video `.mp4` que se
puede ver inline aca abajo, o bajar de Drive.

In [ ]:
!pip install -q imageio imageio-ffmpeg

In [ ]:
import os

import imageio
import numpy as np
import pybullet as p
from IPython.display import Video
from stable_baselines3 import PPO

from quad_env import QuadrupedEnv


def capture_frame(env, width=480, height=360):
    base_pos, _ = p.getBasePositionAndOrientation(env.robot_id, physicsClientId=env.client)
    view = p.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=base_pos, distance=0.6, yaw=50, pitch=-25, roll=0, upAxisIndex=2,
        physicsClientId=env.client,
    )
    proj = p.computeProjectionMatrixFOV(60, width / height, 0.05, 5.0, physicsClientId=env.client)
    _, _, rgb, _, _ = p.getCameraImage(
        width, height, view, proj, renderer=p.ER_TINY_RENDERER, physicsClientId=env.client,
    )
    return np.reshape(rgb, (height, width, 4))[:, :, :3]


final_path = "checkpoints/ppo_quadruped_final.zip"
model_path = final_path if os.path.exists(final_path) else latest_checkpoint()
print("Usando modelo:", model_path)

env = QuadrupedEnv(render_mode=None)
model = PPO.load(model_path, env=env)

obs, _info = env.reset(seed=0)
frames = []
for i in range(600):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if i % 2 == 0:
        frames.append(capture_frame(env))
    if terminated or truncated:
        break
env.close()

video_path = "gait_preview.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Video guardado en {os.path.join(os.getcwd(), video_path)} (dentro de tu Drive)")
Video(video_path, embed=True)